In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.layers import LSTM,Dense,Dropout,Embedding
from tensorflow.keras.optimizers import RMSprop
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences

In [2]:
data = pd.read_csv("/content/spam.csv",encoding='latin-1')
data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
data['v1'].value_counts()

ham     4825
spam     747
Name: v1, dtype: int64

In [4]:
x = data['v2']
y = data['v1']
le = LabelEncoder()
y = le.fit_transform(y)
y = y.reshape(-1,1)
y

array([[0],
       [0],
       [1],
       ...,
       [0],
       [0],
       [0]])

In [5]:
xtrain,xtest,ytrain,ytest = train_test_split(x,y,train_size=0.85,random_state=7)

In [6]:
print(xtrain.shape)
print(ytrain.shape)
print(xtest.shape)
print(ytest.shape)

(4736,)
(4736, 1)
(836,)
(836, 1)


In [7]:
tok = Tokenizer(num_words=1000)
tok.fit_on_texts(xtrain)
sequences = tok.texts_to_sequences(xtrain)
sequences_mat = pad_sequences(sequences,maxlen=150,padding="pre")
sequences_mat

array([[  0,   0,   0, ...,   0, 155, 447],
       [  0,   0,   0, ...,   2,  28, 113],
       [  0,   0,   0, ...,  38,  44,   4],
       ...,
       [  0,   0,   0, ...,  52,   2,  28],
       [  0,   0,   0, ..., 193, 249,  97],
       [  0,   0,   0, ...,   2,  31, 531]], dtype=int32)

In [ ]:
sequences_mat.shape

(4736, 150)

In [8]:
from keras.models import Sequential
model = Sequential()
model.add(Embedding(1000,10,input_length=150))
model.add(LSTM(64))
model.add(Dense(256,activation='relu'))  ## hidden layer
model.add(Dropout(0.5))
model.add(Dense(1,activation="sigmoid"))
model.compile(loss='binary_crossentropy',optimizer=RMSprop(),metrics=['accuracy'])

In [9]:
model.fit(sequences_mat,ytrain,batch_size=64,epochs=15)

Epoch 1/15
74/74 [==============================] - 17s 125ms/step - loss: 0.3433 - accuracy: 0.8824
Epoch 2/15
74/74 [==============================] - 5s 67ms/step - loss: 0.1005 - accuracy: 0.9764
Epoch 3/15
74/74 [==============================] - 4s 56ms/step - loss: 0.0524 - accuracy: 0.9854
Epoch 4/15
74/74 [==============================] - 2s 24ms/step - loss: 0.0417 - accuracy: 0.9871
Epoch 5/15
74/74 [==============================] - 1s 15ms/step - loss: 0.0372 - accuracy: 0.9901
Epoch 6/15
74/74 [==============================] - 1s 17ms/step - loss: 0.0296 - accuracy: 0.9905
Epoch 7/15
74/74 [==============================] - 1s 15ms/step - loss: 0.0255 - accuracy: 0.9935
Epoch 8/15
74/74 [==============================] - 1s 17ms/step - loss: 0.0224 - accuracy: 0.9935
Epoch 9/15
74/74 [==============================] - 1s 9ms/step - loss: 0.0187 - accuracy: 0.9941
Epoch 10/15
74/74 [==============================] - 1s 13ms/step - loss: 0.0167 - accuracy: 0.9960
Epoch 11

In [10]:
text_sequences = tok.texts_to_sequences(xtest)
test_sequences_mat = pad_sequences(text_sequences,maxlen=150,padding="pre")

In [11]:
model.evaluate(test_sequences_mat,ytest)

27/27 [==============================] - 1s 7ms/step - loss: 0.0682 - accuracy: 0.9868


[0.06815817207098007, 0.9868420958518982]

In [ ]:
ypred = model.predict(test_sequences_mat)
ypred = ypred.round()
ypred

In [13]:
from sklearn.metrics import confusion_matrix,accuracy_score
cm = confusion_matrix(ytest,ypred)
ac = accuracy_score(ytest,ypred)

In [14]:
cm

array([[719,   4],
       [  7, 106]])

In [15]:
ac

0.9868421052631579

In [16]:
### Test a new sample
txt = ["You got a 50% discount and free of cost hampers in every purchase",
       "Congratulations, you are selected for the applied job"]
txt = tok.texts_to_sequences(txt)
txt = pad_sequences(txt,maxlen=150,padding="pre")
model.predict(txt).round()

1/1 [==============================] - 0s 103ms/step


array([[1.],
       [0.]], dtype=float32)